# 3 · GRPO / RLAIF (GPU job, DDP)
Optimizes summary quality on top of the SFT model using the **training judge** as the reward. Launches `src/grpo.sh` under torchrun.

> The job role must allow `bedrock:InvokeModel` on `TRAIN_JUDGE_MODEL`. Each rollout is a judge call — `n_grpo` is kept modest and calls are parallelized by `JUDGE_MAX_CONCURRENCY`.

In [ ]:
import json
from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import Session
from sagemaker.train.model_trainer import ModelTrainer
from sagemaker.train.configs import SourceCode, Compute, StoppingCondition

sess = Session()
cfg = json.load(open('summ_config.json'))
region = cfg['region']
INSTANCE = cfg['gpu_instance']
image_uri = cfg['gpu_image_uri'] or image_uris.retrieve(
    framework='pytorch', region=region, version=cfg['pt_version'],
    instance_type=INSTANCE, image_scope='training')

env = {
    'BASE_MODEL': cfg['base_model'], 'SFT_MODEL_S3': cfg['sft_model_s3'],
    'DATA_S3': cfg['data_s3'], 'OUTPUT_GRPO_S3': cfg['grpo_model_s3'],
    'SEQ_LEN': str(cfg['seq_len']), 'SEED': str(cfg['seed']), 'ATTN_IMPL': cfg['attn_impl'],
    'USE_LORA': str(cfg['use_lora']), 'LORA_R': str(cfg['lora_r']), 'LORA_ALPHA': str(cfg['lora_alpha']),
    'GRPO_EPOCHS': str(cfg['grpo_epochs']), 'GRPO_LR': str(cfg['grpo_lr']),
    'GRPO_BS': str(cfg['grpo_bs']), 'GRPO_GRAD_ACCUM': str(cfg['grpo_grad_accum']),
    'NUM_GENERATIONS': str(cfg['num_generations']), 'GRPO_BETA': str(cfg['grpo_beta']),
    'GRPO_TEMPERATURE': str(cfg['grpo_temperature']), 'MAX_COMPLETION_LENGTH': str(cfg['max_completion_length']),
    'JUDGE_BACKEND': cfg['judge_backend'], 'TRAIN_JUDGE_MODEL': cfg['train_judge_model'],
    'JUDGE_REGION': cfg.get('judge_region', '') or region, 'JUDGE_MAX_CONCURRENCY': str(cfg['judge_max_concurrency']),
    'REWARD_WEIGHTS': cfg['reward_weights'],
    'REWARD_MODE': cfg.get('reward_mode', 'guarded'), 'GUARD_WEIGHTS': cfg.get('guard_weights', '0.45,0.30,0.25'),
    'FAITH_GATE': str(cfg.get('faith_gate', 0.40)), 'GATE_MULT': str(cfg.get('gate_mult', 0.30)),
    'COMPARE_AGAINST': cfg.get('compare_against', 'sft'),
    'HF_HOME': '/opt/ml/hf_home', 'HF_HUB_ENABLE_HF_TRANSFER': '1', 'TOKENIZERS_PARALLELISM': 'false',
}
if cfg.get('hf_token'):
    env['HF_TOKEN'] = cfg['hf_token']; env['HUGGING_FACE_HUB_TOKEN'] = cfg['hf_token']
source_code = SourceCode(source_dir='../src', requirements='requirements.txt', entry_script='grpo.sh')
ModelTrainer(sagemaker_session=sess, role=cfg['role'], training_image=image_uri,
    source_code=source_code, compute=Compute(instance_type=INSTANCE, instance_count=1, keep_alive_period_in_seconds=cfg.get('keep_alive', 1800)),
    environment=env, base_job_name='summ-grpo',
    stopping_condition=StoppingCondition(max_runtime_in_seconds=24*3600)).train(wait=True)
print('SFT+GRPO model ->', cfg['grpo_model_s3'])

**Next:** `04_evaluate.ipynb`.